# QUELL Step 06 — Nicemleme + deploy edilebilir model kaydi (YPB)

Qwen2.5-1.5B / Edge'i yeniden egitir, **LoRA adaptorunu kaydeder + zip'ler** (Jetson'a tasinacak), sonra **FP16 ve INT8** dogruluk korunumunu olcer. Kernel RESTART -> Blok 1->2->3.
Cikti: models/..._adapter.zip (transfer) + results/quant_report.json. Blok 2/3 KIYAS'ini paylas.

In [ ]:
# ===== BLOCK 1/3: preparation + model (4-bit QLoRA) =====
import os, subprocess
try:
    _o=subprocess.check_output("nvidia-smi --query-gpu=index,memory.free --format=csv,noheader,nounits",shell=True,text=True)
    _f=[(int(x.split(",")[0]),int(x.split(",")[1])) for x in _o.strip().splitlines()]
    _b=max(_f,key=lambda t:t[1]); os.environ["CUDA_VISIBLE_DEVICES"]=str(_b[0])
    os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
    print("selected GPU:",_b[0],"| free(MiB):",_f,flush=True)
except Exception as e: print("GPU secim atlandi:",e)
import json, time, sys, glob, shutil
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
for pk in ["transformers","peft","accelerate","bitsandbytes"]:
    try: __import__(pk)
    except Exception: subprocess.run([sys.executable,"-m","pip","install","-q",pk])
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

MODEL="Qwen/Qwen2.5-1.5B"; DATASET="edge_iiotset"; TRAIN_CAP=2000; EPOCHS=2; SEED=42; EVAL_CAP=60000
torch.manual_seed(SEED); np.random.seed(SEED)
ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
MODELS=ROOT/"models"; MODELS.mkdir(exist_ok=True)
TAG=f"{DATASET}_{MODEL.split('/')[-1]}"
ADAPTER_DIR=MODELS/f"{TAG}_adapter"; FP16_DIR=MODELS/f"{TAG}_merged_fp16"
rep=json.load(open(RES/"split_report.json")); meta=rep[DATASET]
label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
df=pd.read_parquet(PROC/f"{DATASET}.parquet").reset_index(drop=True)
sp=np.load(SPL/f"{DATASET}_split.npz"); tr_idx,te_idx=sp["train"],sp["test"]
drop=set([label])|set(meta.get("leaky_candidates",[]))
if group: drop.add(group)
if tcol: drop.add(tcol)
for c in df.columns:
    if c!=label and c.lower() in LABELISH: drop.add(c)
feats=[c for c in df.columns if c not in drop]
def row_to_text(r):
    parts=[]
    for c in feats:
        v=r[c]
        if isinstance(v,(float,np.floating)): v=round(float(v),4)
        parts.append(f"{c}={v}")
    return "Network traffic flow. " + ", ".join(parts) + " . Attack type:"
texts_all=df.apply(row_to_text,axis=1).values; y_all=df[label].astype(str).values
rng=np.random.default_rng(SEED); tr_sel=[]
for cls in pd.unique(y_all[tr_idx]):
    ids=tr_idx[y_all[tr_idx]==cls]
    if len(ids)>TRAIN_CAP: ids=rng.choice(ids,TRAIN_CAP,replace=False)
    tr_sel+=ids.tolist()
tr_sel=np.array(sorted(tr_sel))
le=LabelEncoder().fit(y_all[tr_sel]); K=len(le.classes_); classes_all=sorted(pd.unique(y_all).tolist())
MAX_LEN = 256 if len(feats)<=64 else 512
# evaluation indices (full or 60k stratified)
if len(te_idx)<=EVAL_CAP: eval_idx=te_idx
else:
    r2=np.random.default_rng(SEED); pick=[]
    for cls in np.unique(y_all[te_idx]):
        ids=te_idx[y_all[te_idx]==cls]; k=max(1,int(round(len(ids)*EVAL_CAP/len(te_idx))))
        pick+=r2.choice(ids,min(k,len(ids)),replace=False).tolist()
    eval_idx=np.array(sorted(pick))
print(f"{MODEL} | {DATASET}: class={K} train={len(tr_sel):,} eval={len(eval_idx):,} MAX_LEN={MAX_LEN}",flush=True)
tok=AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token=tok.eos_token
class DS(torch.utils.data.Dataset):
    def __init__(self,idx): self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=self.idx[i]; enc=tok(texts_all[j],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
        it={k:v.squeeze(0) for k,v in enc.items()}; it["labels"]=torch.tensor(int(le.transform([y_all[j]])[0])); return it
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
base=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,quantization_config=bnb,device_map={"":0})
base=prepare_model_for_kbit_training(base); base.config.pad_token_id=tok.pad_token_id
model=get_peft_model(base,LoraConfig(task_type=TaskType.SEQ_CLS,r=16,lora_alpha=32,lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],modules_to_save=["score"]))
model.print_trainable_parameters(); print("BLOCK 1 done",flush=True)

In [ ]:
# ===== BLOCK 2/3: train + SAVE THE ADAPTER (+zip) =====
from transformers import TrainingArguments, Trainer
args=TrainingArguments(output_dir=str(MODELS/f"{TAG}_train"),per_device_train_batch_size=8,
    gradient_accumulation_steps=2,num_train_epochs=EPOCHS,learning_rate=2e-4,bf16=True,
    gradient_checkpointing=True,logging_steps=50,save_strategy="no",report_to=[],seed=SEED)
model.config.use_cache=False
tr=Trainer(model=model,args=args,train_dataset=DS(tr_sel))
t=time.time(); tr.train(); print(f"training: {time.time()-t:.0f}s",flush=True)
model.save_pretrained(str(ADAPTER_DIR)); tok.save_pretrained(str(ADAPTER_DIR))
# also save the label mapping (needed on the Jetson)
json.dump({"classes":list(le.classes_),"label_col":label,"feats":feats,"max_len":MAX_LEN,"base_model":MODEL},
          open(ADAPTER_DIR/"quell_meta.json","w"),ensure_ascii=False,indent=2)
zipp=shutil.make_archive(str(MODELS/f"{TAG}_adapter"),"zip",str(ADAPTER_DIR))
sz=os.path.getsize(zipp)/1e6
print(f"BLOCK 2 done. adapter -> {ADAPTER_DIR}  | zip: {zipp} ({sz:.1f} MB)  <-- Jetson'a bunu tasi",flush=True)

In [ ]:
# ===== BLOCK 3/3: FP16 vs INT8 accuracy retention (quantization) =====
import gc
def evaluate(m):
    m.eval(); dev=next(m.parameters()).device; preds=[]; bs=64
    for s in range(0,len(eval_idx),bs):
        js=eval_idx[s:s+bs]
        enc=tok(list(texts_all[js]),truncation=True,max_length=MAX_LEN,padding=True,return_tensors="pt").to(dev)
        with torch.no_grad(), torch.autocast(device_type="cuda",dtype=torch.bfloat16,enabled=(dev.type=="cuda")):
            lo=m(**enc).logits
        preds+=lo.argmax(-1).cpu().tolist()
        if (s//bs)%40==0: print(f"    ...{s:,}/{len(eval_idx):,}",flush=True)
    p=le.inverse_transform(np.array(preds)); yt=y_all[eval_idx]
    return round(accuracy_score(yt,p),4), round(f1_score(yt,p,average="macro",labels=classes_all,zero_division=0),4)

# free the memory, release the training model
del model, base, tr; gc.collect(); torch.cuda.empty_cache()

res={}
# --- FP16 (adaptoru fp16 tabana uygula, birlestir, kaydet) ---
print("Evaluating FP16...",flush=True)
b16=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,torch_dtype=torch.float16,device_map={"":0})
b16.config.pad_token_id=tok.pad_token_id
m16=PeftModel.from_pretrained(b16,str(ADAPTER_DIR)); m16=m16.merge_and_unload()
res["fp16"]=dict(zip(["acc","macro_f1"],evaluate(m16)))
m16.save_pretrained(str(FP16_DIR)); tok.save_pretrained(str(FP16_DIR))
print("FP16 merged model ->",FP16_DIR,flush=True)
del b16,m16; gc.collect(); torch.cuda.empty_cache()

# --- INT8 (bitsandbytes 8-bit taban + adapter) ---
print("Evaluating INT8...",flush=True)
bnb8=BitsAndBytesConfig(load_in_8bit=True)
b8=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=K,quantization_config=bnb8,device_map={"":0})
b8.config.pad_token_id=tok.pad_token_id
m8=PeftModel.from_pretrained(b8,str(ADAPTER_DIR))
res["int8"]=dict(zip(["acc","macro_f1"],evaluate(m8)))
del b8,m8; gc.collect(); torch.cuda.empty_cache()

out=RES/"quant_report.json"; allq=json.load(open(out)) if out.exists() else {}
allq.setdefault(DATASET,{})[MODEL]=res
json.dump(allq,open(out,"w"),indent=2,ensure_ascii=False)
print("\n===== QUANTIZATION RETENTION (macro-F1 / acc) =====")
for prec,v in res.items(): print(f"  {prec:5s}  macroF1={v['macro_f1']}  acc={v['acc']}")
try:
    rf=json.load(open(RES/"baseline_report.json"))[DATASET]["models"]["random_forest"]["macro_f1"]
    print(f"  (referans RF macroF1={rf})")
except: pass
print("saved -> results/quant_report.json | BLOCK 3 done",flush=True)